<a href="https://colab.research.google.com/github/Luca4Spreafico/CHALLENGE-2---Ibuprofen/blob/main/Swin_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Swin Transformer

# Google colab:

In [3]:
from google.colab import drive

import os
import shutil
import numpy as np
from PIL import Image
import pandas as pd
from pathlib import Path
!pip install lion-pytorch
# Define your working directory
drive.mount("/gdrive")
working_dir = "/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2"
%cd $working_dir



Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2


# Train & Val

In [10]:
"""
Swin Transformer for Breast Cancer Molecular Subtype Classification
Handles pre-cropped 512x512 tiles with multiple crops per original image
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import torch.optim as optim
from lion_pytorch import Lion

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Label mapping
LABEL_MAP = {
    'Luminal A': 0,
    'Luminal B': 1,
    'HER2(+)': 2,
    'Triple negative': 3
}

LABEL_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple Negative']


class BreastCancerCropDataset(Dataset):
    """
    Dataset for pre-cropped breast cancer histopathology images
    Handles multiple crops from the same original image
    """
    def __init__(self, csv_file, img_dir, transform=None, label_col='label'):
        """
        Args:
            csv_file: Path to CSV with columns [sample_index, label]
            img_dir: Directory containing the cropped images
            transform: Optional transform to apply
            label_col: Name of the label column in CSV
        """
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.label_col = label_col

        # Map string labels to integers if needed
        if self.df[label_col].dtype == 'object':
            self.df['label_int'] = self.df[label_col].map(LABEL_MAP)
        else:
            self.df['label_int'] = self.df[label_col]

        print(f"Loaded {len(self.df)} crops")
        print(f"Class distribution:\n{self.df[label_col].value_counts()}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get image filename and label
        img_name = self.df.iloc[idx]['sample_index']
        label = self.df.iloc[idx]['label_int']

        # Load image
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return image, label, img_name


class SwinBreastCancerClassifier(nn.Module):
    """
    Swin Transformer model for breast cancer subtype classification
    """
    def __init__(self, num_classes=4, pretrained=True, model_name='swin_base_patch4_window12_384'):
        super(SwinBreastCancerClassifier, self).__init__()

        # Load pretrained Swin Transformer
        self.model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes
        )

        print(f"Created {model_name} with {num_classes} classes")
        print(f"Pretrained: {pretrained}")

    def forward(self, x):
        return self.model(x)


def get_transforms(img_size=384, augment=True):
    """
    Get training and validation transforms
    """
    if augment:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform


def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in pbar:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/len(pbar):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, labels, _ in pbar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'loss': f'{running_loss/len(pbar):.4f}',
                'acc': f'{100*correct/total:.2f}%'
            })

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels


def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES,
                yticklabels=LABEL_NAMES)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_training_history(train_losses, train_accs, val_losses, val_accs, save_path='training_history.png'):
    """Plot training history"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(train_losses, label='Train Loss', marker='o')
    ax1.plot(val_losses, label='Val Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot([acc*100 for acc in train_accs], label='Train Acc', marker='o')
    ax2.plot([acc*100 for acc in val_accs], label='Val Acc', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Training history saved to {save_path}")


def main():
    """Main training pipeline"""

    # Configuration
    CONFIG = {
        'csv_file': 'train_labels_crops_crop512_ov128 (1).csv',
        'img_dir': 'train_data_crops_crop512_ov128',
        'img_size': 384,  # Swin Transformer typically uses 384
        'batch_size': 16,
        'num_epochs': 30,
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'num_workers': 4,
        'model_name': 'swin_base_patch4_window12_384',  # or 'swin_tiny_patch4_window7_224'
        'save_dir': 'models',
      }

    CONFIG_LION = {
      'learning_rate': 1e-5,      # 10x smaller than AdamW
      'weight_decay': 0.5,         # 10x larger than AdamW
      'beta1': 0.9,                # Default
      'beta2': 0.99,                # Default
      }

    # Create save directory
    os.makedirs(CONFIG['save_dir'], exist_ok=True)

    print("="*50)
    print("Swin Transformer Training Configuration")
    print("="*50)
    for key, value in CONFIG.items():
        print(f"{key}: {value}")
    print("="*50)

    # Get transforms
    train_transform, val_transform = get_transforms(
        img_size=CONFIG['img_size'],
        augment=True
    )

    # Load full dataset
    full_dataset = BreastCancerCropDataset(
        csv_file=CONFIG['csv_file'],
        img_dir=CONFIG['img_dir'],
        transform=None,
        label_col='label'  # ← ADD THIS LINE
    )

    # Split by original image (not by crop) to avoid data leakage
    # Extract original image IDs from filenames
    full_dataset.df['orig_img_id'] = full_dataset.df['sample_index'].str.extract(r'(img_\d+)')[0]

    # Get unique image IDs and their labels
    unique_imgs = full_dataset.df.groupby('orig_img_id')['label_int'].first().reset_index()

    # Split unique images
    train_imgs, val_imgs = train_test_split(
        unique_imgs['orig_img_id'].values,
        test_size=0.2,
        stratify=unique_imgs['label_int'].values,
        random_state=42
    )

    print(f"\nSplit by original images:")
    print(f"Train images: {len(train_imgs)}")
    print(f"Val images: {len(val_imgs)}")

    # Create train and val dataframes
    train_df = full_dataset.df[full_dataset.df['orig_img_id'].isin(train_imgs)].reset_index(drop=True)
    val_df = full_dataset.df[full_dataset.df['orig_img_id'].isin(val_imgs)].reset_index(drop=True)

    print(f"\nCrops per split:")
    print(f"Train crops: {len(train_df)}")
    print(f"Val crops: {len(val_df)}")

    # Create datasets
    train_dataset = BreastCancerCropDataset(
        csv_file=CONFIG['csv_file'],
        img_dir=CONFIG['img_dir'],
        transform=train_transform,
        label_col='label'  # ← ADD THIS LINE
    )
    train_dataset.df = train_df

    val_dataset = BreastCancerCropDataset(
        csv_file=CONFIG['csv_file'],
        img_dir=CONFIG['img_dir'],
        transform=val_transform,
        label_col='label'  # ← ADD THIS LINE
    )
    val_dataset.df = val_df

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    # Create model
    model = SwinBreastCancerClassifier(
        num_classes=4,
        pretrained=True,
        model_name=CONFIG['model_name']
    ).to(device)

    '''
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    '''

    criterion = nn.CrossEntropyLoss()
    optimizer = Lion(
      model.parameters(),
      lr=CONFIG_LION['learning_rate'],
      weight_decay=CONFIG_LION['weight_decay'],
      betas=(CONFIG_LION['beta1'], CONFIG_LION['beta2'])
    )

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG['num_epochs']
    )

    # Training loop
    best_val_acc = 0.0
    best_val_f1 = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []

    print("\n" + "="*50)
    print("Starting Training")
    print("="*50)

    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        print("-" * 50)

        # Train
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validate
        val_loss, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, device
        )

        # Update scheduler
        scheduler.step()

        # Store metrics
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        val_f1_macro = f1_score(val_labels, val_preds, average='macro')
        val_f1_weighted = f1_score(val_labels, val_preds, average='weighted')
        print(f"Val F1-Score (macro): {val_f1_macro:.4f}")
        print(f"Val F1-Score (weighted): {val_f1_weighted:.4f}")

        # Save best model
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, os.path.join(CONFIG['save_dir'], 'best_model.pth'))
            print(f"✓ New best model saved! Val Acc: {val_acc*100:.2f}%")

        # Save checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, os.path.join(CONFIG['save_dir'], f'checkpoint_epoch_{epoch+1}.pth'))

    # Final evaluation
    print("\n" + "="*50)
    print("Training Complete!")
    print("="*50)
    print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")

    # Load best model for final evaluation
    checkpoint = torch.load(os.path.join(CONFIG['save_dir'], 'best_model.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])

    # Final validation
    _, final_acc, final_preds, final_labels = validate(
        model, val_loader, criterion, device
    )

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(
        final_labels,
        final_preds,
        target_names=LABEL_NAMES,
        digits=4
    ))

    # Plot confusion matrix
    plot_confusion_matrix(
        final_labels,
        final_preds,
        save_path=os.path.join(CONFIG['save_dir'], 'confusion_matrix.png')
    )

    # Plot training history
    plot_training_history(
        train_losses, train_accs, val_losses, val_accs,
        save_path=os.path.join(CONFIG['save_dir'], 'training_history.png')
    )

    print("\n✓ All results saved to:", CONFIG['save_dir'])


if __name__ == '__main__':
    main()

Using device: cuda
Swin Transformer Training Configuration
csv_file: train_labels_crops_crop512_ov128 (1).csv
img_dir: train_data_crops_crop512_ov128
img_size: 384
batch_size: 16
num_epochs: 30
learning_rate: 0.0001
weight_decay: 0.0001
num_workers: 4
model_name: swin_base_patch4_window12_384
save_dir: models
Loaded 1859 crops
Class distribution:
label
Luminal B          666
Luminal A          505
HER2(+)            474
Triple negative    214
Name: count, dtype: int64

Split by original images:
Train images: 464
Val images: 117

Crops per split:
Train crops: 1496
Val crops: 363
Loaded 1859 crops
Class distribution:
label
Luminal B          666
Luminal A          505
HER2(+)            474
Triple negative    214
Name: count, dtype: int64
Loaded 1859 crops
Class distribution:
label
Luminal B          666
Luminal A          505
HER2(+)            474
Triple negative    214
Name: count, dtype: int64
Created swin_base_patch4_window12_384 with 4 classes
Pretrained: True

Starting Training

E

Validation: 100%|██████████| 23/23 [00:03<00:00,  5.97it/s, loss=1.2921, acc=35.26%]



Epoch 1 Summary:
Train Loss: 1.3165 | Train Acc: 35.23%
Learning Rate: 0.000010
Val F1-Score (macro): 0.3222
Val F1-Score (weighted): 0.3470
✓ New best model saved! Val Acc: 35.26%

Epoch 2/30
--------------------------------------------------


Validation: 100%|██████████| 23/23 [00:03<00:00,  5.92it/s, loss=1.3127, acc=34.44%]



Epoch 2 Summary:
Train Loss: 1.2174 | Train Acc: 44.99%
Learning Rate: 0.000010
Val F1-Score (macro): 0.3196
Val F1-Score (weighted): 0.3273

Epoch 3/30
--------------------------------------------------


...


Epoch 29 Summary:
Train Loss: 0.0117 | Train Acc: 99.80%
Learning Rate: 0.000000
Val F1-Score (macro): 0.3555
Val F1-Score (weighted): 0.3757

Epoch 30/30
--------------------------------------------------


Validation: 100%|██████████| 23/23 [00:03<00:00,  5.93it/s, loss=4.5843, acc=37.74%]



Epoch 30 Summary:
Train Loss: 0.0226 | Train Acc: 99.40%
Learning Rate: 0.000000
Val F1-Score (macro): 0.3610
Val F1-Score (weighted): 0.3763

Training Complete!
Best Validation Accuracy: 0.00%


Validation: 100%|██████████| 23/23 [00:05<00:00,  4.03it/s, loss=3.2012, acc=41.60%]



Classification Report:
                 precision    recall  f1-score   support

      Luminal A     0.3359    0.4388    0.3805        98
      Luminal B     0.4954    0.3942    0.4390       137
        HER2(+)     0.4409    0.4556    0.4481        90
Triple Negative     0.3939    0.3421    0.3662        38

       accuracy                         0.4160       363
      macro avg     0.4165    0.4076    0.4085       363
   weighted avg     0.4282    0.4160    0.4179       363

Confusion matrix saved to models/confusion_matrix.png
Training history saved to models/training_history.png

✓ All results saved to: models


# Testing

In [11]:
"""
Complete Test Pipeline for Breast Cancer Subtype Classification
1. Crops test images (512x512, overlap 128) - saves to test_crops_512_128/
2. Loads trained model
3. Predicts and aggregates
4. Saves submission.csv
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from tqdm import tqdm
from collections import defaultdict

# ============================================================================
# PART 1: CROPPING TEST IMAGES
# ============================================================================

def is_tissue(tile, threshold=0.7):
    """Check if tile contains tissue (not background)"""
    gray = np.array(tile.convert('L'))
    white_ratio = np.sum(gray > 200) / gray.size
    return white_ratio < threshold


def crop_test_images(input_dir, output_dir, tile_size=512, overlap=128):
    """
    Crop test images into tiles
    Saves to output_dir for future reuse
    """
    print("="*60)
    print("STEP 1: CROPPING TEST IMAGES")
    print("="*60)

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Check if already cropped
    metadata_path = os.path.join(output_dir, 'metadata.csv')
    if os.path.exists(metadata_path):
        print(f"✓ Crops already exist in {output_dir}")
        metadata = pd.read_csv(metadata_path)
        print(f"✓ Found {len(metadata)} existing crops")
        return metadata

    # Get image files
    image_files = [f for f in os.listdir(input_dir)
                   if f.lower().endswith(('.png'))]

    print(f"Found {len(image_files)} test images to crop")
    print(f"Tile size: {tile_size}x{tile_size}, Overlap: {overlap}")

    metadata = []
    stride = tile_size - overlap
    total_crops = 0

    for img_name in tqdm(image_files, desc="Cropping"):
        img_path = os.path.join(input_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        width, height = image.size
        base_name = os.path.splitext(img_name)[0]

        crop_idx = 0
        for y in range(0, height - tile_size + 1, stride):
            for x in range(0, width - tile_size + 1, stride):
                tile = image.crop((x, y, x + tile_size, y + tile_size))

                # Only save tiles with tissue
                if is_tissue(tile, threshold=0.7):
                    crop_name = f"{base_name}_{crop_idx:03d}.png"
                    tile.save(os.path.join(output_dir, crop_name))

                    metadata.append({
                        'crop_filename': crop_name,
                        'original_image': img_name,
                        'original_basename': base_name
                    })
                    crop_idx += 1
                    total_crops += 1

    # Save metadata
    metadata_df = pd.DataFrame(metadata)
    metadata_df.to_csv(metadata_path, index=False)

    print(f"✓ Created {total_crops} crops from {len(image_files)} images")
    print(f"✓ Average crops per image: {total_crops/len(image_files):.1f}")
    print(f"✓ Saved to: {output_dir}")
    print(f"✓ Metadata saved to: {metadata_path}")

    return metadata_df


# ============================================================================
# PART 2: MODEL AND DATASET
# ============================================================================

class TestDataset(Dataset):
    """Dataset for test crops"""
    def __init__(self, crop_dir, metadata_df, transform):
        self.crop_dir = crop_dir
        self.metadata = metadata_df
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        crop_path = os.path.join(self.crop_dir, row['crop_filename'])
        image = Image.open(crop_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, row['crop_filename'], row['original_basename']


class SwinModel(nn.Module):
    """Swin Transformer model"""
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = timm.create_model(
            'swin_base_patch4_window12_384',
            pretrained=False,
            num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)


# ============================================================================
# PART 3: PREDICTION AND AGGREGATION
# ============================================================================

def predict_crops(model, loader, device):
    """Predict all crops"""
    model.eval()
    predictions = []

    with torch.no_grad():
        for images, crop_names, orig_names in tqdm(loader, desc="Predicting"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            for prob, crop_name, orig_name in zip(probs, crop_names, orig_names):
                predictions.append({
                    'crop_filename': crop_name,
                    'original_basename': orig_name,
                    'probabilities': prob.cpu().numpy()
                })

    return predictions


def aggregate_predictions(predictions, label_names):
    """Aggregate crop predictions to image-level predictions"""
    print("\n" + "="*60)
    print("AGGREGATING PREDICTIONS")
    print("="*60)

    # Group by original image
    image_groups = defaultdict(list)
    for pred in predictions:
        image_groups[pred['original_basename']].append(pred['probabilities'])

    results = []
    for img_basename, probs_list in image_groups.items():
        # Average probabilities across all crops
        avg_probs = np.mean(probs_list, axis=0)
        pred_class = np.argmax(avg_probs)
        confidence = avg_probs[pred_class]

        results.append({
            'sample_index': img_basename + '.png',
            'label': label_names[pred_class],
            'confidence': confidence,
            'num_crops': len(probs_list)
        })

    results_df = pd.DataFrame(results)

    print(f"Aggregated {len(results_df)} images")
    print(f"\nPrediction distribution:")
    print(results_df['label'].value_counts())
    print(f"\nAverage confidence: {results_df['confidence'].mean():.4f}")
    print(f"Average crops per image: {results_df['num_crops'].mean():.1f}")

    return results_df


def save_submission(results_df, output_path):
    """Save submission file"""
    submission = results_df[['sample_index', 'label']].copy()
    submission.to_csv(output_path, index=False)
    print(f"\n✓ Submission saved to: {output_path}")
    return submission


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Complete test pipeline"""

    # Configuration
    CONFIG = {
        'test_images_dir': 'organized_data/test_images',
        'crop_output_dir': 'test_crops_512_128',  # Reusable crop folder
        'model_path': 'models/best_model.pth',
        'submission_path': 'models/submission.csv',
        'batch_size': 32,
        'num_workers': 4,
        'img_size': 384,
    }

    LABEL_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    # ========================================================================
    # STEP 1: Crop test images (or load existing crops)
    # ========================================================================
    '''
    metadata_df = crop_test_images(
        input_dir=CONFIG['test_images_dir'],
        output_dir=CONFIG['crop_output_dir'],
        tile_size=512,
        overlap=128
    )
    '''
    # ============================================================================
    # CREATE METADATA FROM EXISTING CROPS
    # ============================================================================

    crop_files = sorted([f for f in os.listdir(CONFIG['crop_output_dir']) if f.endswith('.png')])

    # Extract original basename from crop filename
    # Example: img_0000_001.png -> img_0000
    original_basenames = [f.rsplit('_', 1)[0] for f in crop_files]

    metadata_df = pd.DataFrame({
        'crop_filename': crop_files,
        'original_basename': original_basenames,  # Add this column
        'original_image': [f + '.png' for f in original_basenames]  # Full name with .png
    })
    # ========================================================================
    # STEP 2: Load model
    # ========================================================================
    print("\n" + "="*60)
    print("STEP 2: LOADING MODEL")
    print("="*60)

    model = SwinModel(num_classes=4).to(device)
    checkpoint = torch.load(CONFIG['model_path'], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print(f"✓ Model loaded from: {CONFIG['model_path']}")
    if 'val_acc' in checkpoint:
        print(f"  Training val accuracy: {checkpoint['val_acc']*100:.2f}%")

    # ========================================================================
    # STEP 3: Prepare dataset
    # ========================================================================
    print("\n" + "="*60)
    print("STEP 3: PREPARING TEST DATA")
    print("="*60)

    transform = transforms.Compose([
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    dataset = TestDataset(
        crop_dir=CONFIG['crop_output_dir'],
        metadata_df=metadata_df,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    print(f"✓ Prepared {len(dataset)} crops for prediction")

    # ========================================================================
    # STEP 4: Predict all crops
    # ========================================================================
    print("\n" + "="*60)
    print("STEP 4: MAKING PREDICTIONS")
    print("="*60)

    predictions = predict_crops(model, loader, device)
    print(f"✓ Predicted {len(predictions)} crops")

    # ========================================================================
    # STEP 5: Aggregate and save
    # ========================================================================
    results_df = aggregate_predictions(predictions, LABEL_NAMES)

    submission = save_submission(results_df, CONFIG['submission_path'])

    # Show sample results
    print("\n" + "="*60)
    print("SAMPLE PREDICTIONS")
    print("="*60)
    print(submission.head(10).to_string(index=False))

    print("\n" + "="*60)
    print("✓ TESTING COMPLETE!")
    print("="*60)
    print(f"✓ Crops saved in: {CONFIG['crop_output_dir']} (reusable)")
    print(f"✓ Submission saved: {CONFIG['submission_path']}")
    print(f"✓ Total predictions: {len(submission)}")


if __name__ == '__main__':
    main()

Using device: cuda


STEP 2: LOADING MODEL
✓ Model loaded from: models/best_model.pth
  Training val accuracy: 41.60%

STEP 3: PREPARING TEST DATA
✓ Prepared 3151 crops for prediction

STEP 4: MAKING PREDICTIONS


Predicting: 100%|██████████| 99/99 [00:49<00:00,  1.98it/s]


✓ Predicted 3151 crops

AGGREGATING PREDICTIONS
Aggregated 477 images

Prediction distribution:
label
Luminal A          172
Luminal B          168
HER2(+)            125
Triple negative     12
Name: count, dtype: int64

Average confidence: 0.6080
Average crops per image: 6.6

✓ Submission saved to: models/submission.csv

SAMPLE PREDICTIONS
sample_index           label
img_0000.png       Luminal B
img_0001.png       Luminal B
img_0002.png       Luminal B
img_0003.png         HER2(+)
img_0004.png       Luminal B
img_0005.png       Luminal A
img_0006.png Triple negative
img_0007.png       Luminal A
img_0008.png       Luminal A
img_0009.png       Luminal A

✓ TESTING COMPLETE!
✓ Crops saved in: test_crops_512_128 (reusable)
✓ Submission saved: models/submission.csv
✓ Total predictions: 477
